# Lab 1: Profiling a model and measuring your own roofline
**AIMS Edge ML for Science · Day 2 · ~2 hours**

On Day 1 we said a model is only useful if it fits its **budget**: latency, throughput, power and memory.
Today you measure where a small model actually sits on **your Colab machine**, on both its **CPU** and its **GPU**, and why.

By the end of this lab you will have:
1. Counted a network's parameters, operations and bytes **by hand**, and turned them into an *arithmetic intensity*.
2. Timed inference properly: warm-up, percentiles, threads, batching.
3. Measured your machine's peak compute and memory bandwidth, and drawn its **roofline**, for the CPU *and* the GPU.
4. Placed the model on those rooflines and explained what limits it, including when a GPU helps and when it doesn't.
5. Compared all of this to the LHC Level-1 trigger budget.

| Mark | Meaning |
|---|---|
| ✋ **Pause** | One of three points where we stop and discuss as a class. |
| 📌 **What this shows** | The conclusion from what you just measured. Read it, and check it against your own numbers. |
| 🟥 **AI-off** | Pen and paper. No AI tools. Questions like these will be on the exam. |
| 🟩 **AI-on** | Use whatever tools you like. You must run the code yourself and be able to explain it. |

> **Before you start:**
> 1. **File → Save a copy in Drive**, and work in the copy (otherwise your answers aren't saved).
> 2. **Runtime → Change runtime type → T4 GPU → Save.**
> 3. Run the first code cell (**Shift + Enter**; click *Run anyway* if Colab warns you). Then work down one cell at a time.
>
> Colab machines are **shared cloud computers**, so your timings will be noisier than on a dedicated machine. That's part of today's lesson about tails.

## 0. Setup (5 min)

In [ ]:
# --- Colab setup: run once per session (~30 s) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the dataset on your Google Drive, so later sessions skip the download
LAB = "lab01_profiling"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")            # instructor test bundle, if uploaded
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)      # pick up new releases
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
        os.environ["SCIKIT_LEARN_DATA"] = "/content/drive/MyDrive/fastml_data/sklearn"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import sys, os, time, platform
sys.path.append("..")                      # makes `common/` importable
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from common.jets import load_jets, CLASSES

torch.manual_seed(0); np.random.seed(0)
N_CORES = os.cpu_count()
print(f"Python {platform.python_version()} | torch {torch.__version__} | numpy {np.__version__}")
print(f"{platform.system()} {platform.machine()} | {platform.processor() or '?'} | {N_CORES} logical cores")
print(f"torch threads: {torch.get_num_threads()}")
HAS_GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none. Section 5c needs one: Runtime → Change runtime type → GPU, then rerun from the top.")

Note your CPU and GPU models; you'll need them in Section 5. In a code cell, run `!lscpu | grep "Model name"` and `!nvidia-smi --query-gpu=name,memory.total --format=csv`.

*My CPU:* ______________________  *My GPU:* ______________________

## 1. The model (10 min)
We use the standard fast-ML benchmark: **jet tagging** at the LHC. Each jet is described by 16 high-level features,
and the task is to classify it as coming from a gluon, a light quark, a W, a Z or a top quark.
The model is the small MLP used throughout the hls4ml literature: **16 → 64 → 32 → 32 → 5**, ReLU.

In [ ]:
X_train, X_test, y_train, y_test, features = load_jets()
print(X_train.shape, X_test.shape, "classes:", CLASSES)

class JetMLP(nn.Module):
    def __init__(self, sizes=(16, 64, 32, 32, 5)):
        super().__init__()
        layers = []
        for i, (a, b) in enumerate(zip(sizes[:-1], sizes[1:])):
            layers.append(nn.Linear(a, b))
            if i < len(sizes) - 2:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

model = JetMLP()
print(model)

In [ ]:
# Quick training on the CPU. Takes ~30-60 s. Today is not about accuracy, but we want a real model.
def train(model, X, y, epochs=3, bs=1024, lr=3e-3):
    Xt, yt = torch.from_numpy(X), torch.from_numpy(y)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    for ep in range(epochs):
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), bs):
            idx = perm[i:i + bs]
            opt.zero_grad(); loss = lossf(model(Xt[idx]), yt[idx]); loss.backward(); opt.step()
        print(f"epoch {ep}: loss {loss.item():.3f}")
    return model

t0 = time.perf_counter(); train(model, X_train, y_train); print(f"trained in {time.perf_counter()-t0:.0f} s")
model.eval()
with torch.inference_mode():
    acc = (model(torch.from_numpy(X_test)).argmax(1).numpy() == y_test).mean()
print(f"test accuracy: {acc:.3f}   (the literature value for this model is ~0.75)")

## 2. 🟥 AI-off: the budget on paper (15 min)  ✋ Pause 1
Answer these **on paper first**, then check them with the code below.
Conventions: one multiply-accumulate (MAC) = 2 FLOPs. Ignore biases and ReLUs in the FLOP count, but **include** biases in the parameter count. Use fp32 (4 bytes).

**Q1** For one inference (batch size 1): how many **parameters**, **MACs** (and FLOPs), and **bytes** must be moved? For the bytes, count the weights
(assume they come from main memory every call) plus the input and output; assume intermediate results stay in cache.

**Q2** The *arithmetic intensity* is $I = \text{FLOPs} / \text{bytes}$. What is $I$ at batch size 1? Now write $I(B)$ for batch size $B$, with the weights read
**once per batch**. What value does it approach for very large $B$, and why can't batching push it further?

*Your answers:*


In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

def count_macs(model):
    """MACs for one sample through every nn.Linear layer (ignore biases and activations)."""
    # YOUR CODE HERE
    raise NotImplementedError

def bytes_moved(model, batch, dtype_bytes=4):
    """Bytes for one batch: weights read once, plus input and output of every sample."""
    lin = [m for m in model.modules() if isinstance(m, nn.Linear)]
    # YOUR CODE HERE
    raise NotImplementedError

def intensity(model, batch):
    return 2 * count_macs(model) * batch / bytes_moved(model, batch)

print("params", count_params(model), "| MACs", count_macs(model), "| bytes(B=1)", bytes_moved(model, 1))
for B in [1, 16, 256, 4096, 10**6]:
    print(f"I(B={B:>7}) = {intensity(model, B):6.2f} FLOP/byte")
assert count_macs(model) == 4256 and bytes_moved(model, 1) == 17640

## 3. 🟩 Timing inference properly (20 min)
Timing code looks easy and is easy to get wrong. Start with the naive approach:

In [ ]:
x1 = torch.from_numpy(X_test[:1])
for _ in range(5):
    t0 = time.time(); model(x1); print(f"{(time.time()-t0)*1e6:8.1f} µs")

📌 **What this shows.** The first call is much slower, because PyTorch sets things up the first time it runs (allocating memory, choosing kernels) and the caches start cold.
That's why every serious timing **warms up** first. Calls this short also need a fine-grained clock: use `time.perf_counter_ns()`, not `time.time()`.

Write a proper timing helper. It should:
- run `n_warmup` untimed calls,
- time each of `n_iter` calls with `time.perf_counter_ns()`,
- return an array of per-call times **in microseconds**.

In [ ]:
def measure(fn, n_warmup=50, n_iter=500):
    # YOUR CODE HERE
    raise NotImplementedError

def summary(t, label=""):
    print(f"{label:<28} median {np.median(t):8.2f} µs | p99 {np.percentile(t, 99):8.2f} µs | min {t.min():8.2f} µs")

t_grad = measure(lambda: model(x1))
with torch.inference_mode():
    t_inf = measure(lambda: model(x1))
summary(t_grad, "batch 1, autograd on"); summary(t_inf, "batch 1, inference_mode")

plt.hist(t_inf, bins=80, log=True); plt.xlabel("latency [µs]"); plt.ylabel("count"); plt.title("batch-1 latency distribution"); plt.show()

📌 **What this shows.** Most calls are fast, but a few are much slower: the long right tail. It comes from the operating system, other users of this shared machine,
and cache misses. A trigger must decide on *every* collision, so the slow calls are lost data. That's why real-time systems quote **p99** (or p99.9), not the mean.

Now compare against the same network written directly in NumPy, with no framework:

In [ ]:
W = [(m.weight.detach().numpy().T.copy(), m.bias.detach().numpy().copy()) for m in model.modules() if isinstance(m, nn.Linear)]

def numpy_forward(x):
    for i, (w, b) in enumerate(W):
        x = x @ w + b
        if i < len(W) - 1:
            x = np.maximum(x, 0)
    return x

x1_np = X_test[:1]
assert np.allclose(numpy_forward(x1_np), model(x1).detach().numpy(), atol=1e-5)
summary(measure(lambda: numpy_forward(x1_np)), "batch 1, numpy")

for n in [1, N_CORES]:
    torch.set_num_threads(n)
    with torch.inference_mode():
        summary(measure(lambda: model(x1)), f"batch 1, torch, {n} thread(s)")
torch.set_num_threads(N_CORES)

📌 **What this shows.** The network needs only ~8.5k FLOPs, well under a microsecond of arithmetic on any modern core, yet PyTorch takes tens of µs or more.
Almost all of that is **overhead**: Python and the framework doing bookkeeping for each of the 7 layers. If NumPy came out faster, that's why: it does less bookkeeping.
More threads don't help at batch 1, because there's no parallel work to share, only extra coordination. This is the **overhead-bound** regime: neither compute nor memory is the limit.

## 4. 🟩 Latency vs. throughput: the batch sweep (15 min)
Sweep the batch size, and record the latency per *batch* and the throughput in *samples per second*.

In [ ]:
batches = [1, 4, 16, 64, 256, 1024, 4096, 16384, 65536]
Xbig = torch.from_numpy(np.resize(X_test, (max(batches), 16)).astype(np.float32))
lat, thr = [], []
with torch.inference_mode():
    for B in batches:
        xb = Xbig[:B]
        n_iter = max(5, min(300, int(2e6 / B)))   # keep each point to a fraction of a second
        t = measure(lambda: model(xb), n_warmup=5, n_iter=n_iter)
        lat.append(np.median(t)); thr.append(B / (np.median(t) * 1e-6))
        print(f"B={B:>6}: {lat[-1]:10.1f} µs/batch  {thr[-1]/1e6:8.2f} M samples/s")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].loglog(batches, lat, "o-"); ax[0].set(xlabel="batch size", ylabel="latency per batch [µs]")
ax[1].semilogx(batches, np.array(thr) / 1e6, "o-"); ax[1].set(xlabel="batch size", ylabel="throughput [M samples/s]")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

📌 **What this shows.** On the left, latency stays almost flat over a range of small batches: a batch of 64 costs about as much as a batch of 1, because the fixed per-call
overhead dominates. Only at large batches does the real work take over. So batching gives throughput almost for free, up to a point, but it never makes a *single* answer come back sooner.

**Q3** 🟥 *AI-off.* **Little's law:** (samples in flight) = throughput × latency. A radio-telescope pipeline needs 50 M samples/s and can tolerate 10 ms latency.
A trigger needs 40 M samples/s and **1 µs** latency. Using your plots, which of the two could this machine serve, and how?



## 5. 🟩 The CPU roofline (20 min)
The **roofline model**: attainable FLOP/s $= \min(P_\text{peak},\; I \times \beta)$, where $\beta$ is the memory bandwidth.
Now measure $P_\text{peak}$ and $\beta$ for your Colab machine's **CPU**. (Section 5c repeats this on the GPU.)

**5a. Peak compute.** A large square matmul has a high arithmetic intensity, so it runs close to peak.

In [ ]:
def matmul_gflops(n=2048, reps=5):
    a, b = torch.randn(n, n), torch.randn(n, n)
    # YOUR CODE HERE
    raise NotImplementedError

for n in [256, 1024, 2048]:
    print(f"matmul n={n}: {matmul_gflops(n):8.1f} GFLOP/s")
P_peak = matmul_gflops(2048) * 1e9

**5b. Memory bandwidth.** Use a STREAM-like *triad* `a = b + s*c` on arrays far bigger than any cache.
Count the bytes carefully: two reads and one write per element.

In [ ]:
def triad_gbs(n=32 * 2**20, reps=5):
    b, c = torch.randn(n), torch.randn(n); a = torch.empty(n)
    # YOUR CODE HERE
    raise NotImplementedError

for n in [2**14, 2**17, 2**20, 32 * 2**20]:
    print(f"triad n={n:>10} ({3*4*n/2**20:7.1f} MiB touched): {triad_gbs(n):8.1f} GB/s")
beta = triad_gbs() * 1e9
print(f"\nP_peak ≈ {P_peak/1e9:.0f} GFLOP/s,  β ≈ {beta/1e9:.0f} GB/s,  ridge point ≈ {P_peak/beta:.1f} FLOP/byte")

📌 **What this shows.** The "bandwidth" depends on the array size, because you're measuring *different memories*. Mid-size arrays (a few MiB) fit in the CPU's caches, which are
faster than main memory. The smallest arrays are dominated by per-call overhead again. Only the largest size, far bigger than any cache, measures main-memory bandwidth β, which is the one we use.

Now draw the roofline and place the jet MLP on it at each batch size, using the intensity from Section 2 and the throughput from Section 4.

In [ ]:
I_axis = np.logspace(-2, 3, 200)
B_arr = np.array(batches)
I_mlp = np.array([intensity(model, B) for B in batches])
F_mlp = np.array(thr) * 2 * count_macs(model)           # achieved FLOP/s

plt.figure(figsize=(7, 5))
plt.loglog(I_axis, np.minimum(P_peak, I_axis * beta) / 1e9, "k-", lw=2, label="roofline (measured)")
sc = plt.scatter(I_mlp, F_mlp / 1e9, c=np.log2(B_arr), cmap="viridis", zorder=3, label="jet MLP")
for B, i, f in zip(batches, I_mlp, F_mlp):
    plt.annotate(f"B={B}", (i, f / 1e9), textcoords="offset points", xytext=(5, -10), fontsize=8)
plt.axvline(P_peak / beta, ls=":", c="gray"); plt.text(P_peak / beta * 1.1, P_peak / 1e9 * 0.05, "ridge", color="gray")
plt.xlabel("arithmetic intensity [FLOP/byte]"); plt.ylabel("performance [GFLOP/s]")
plt.title("CPU roofline"); plt.legend(loc="lower right"); plt.grid(alpha=.3, which="both"); plt.show()

📌 **What this shows.** At B = 1 the model sits far *below* both lines of the roofline, often 100× or more: it's neither compute- nor memory-bound but **overhead-bound** (Section 3).
As B grows, the points move right (the weights are reused more) and up, towards the roof. At large B you may still not reach the main-memory roof, and that's expected: the model's
17.6 kB of weights live in cache, so its real limit is cache bandwidth, a higher roof than the one drawn. (Performance tools draw one roof per cache level.)

Ways to lift the batch-1 point, which the rest of the course covers: remove the framework overhead (ONNX Runtime, Lab 3); move fewer bytes and do fewer operations
(quantization and pruning, Labs 2a/2b); or swap instruction-driven hardware for a circuit with the weights built in (FPGAs, Day 7 and Lab 4).

## 5c. 🟩 The same questions on a GPU (15 min)
A GPU has thousands of simple arithmetic lanes and very fast memory. Does that make it better *for this model*? Measure it.

**One thing is different about timing a GPU.** When Python launches work on the GPU, the call returns *immediately*, while the GPU is still working.
Timing the Python call would only measure the launch. `measure_gpu` below therefore waits for the GPU to finish (`torch.cuda.synchronize()`) inside every
timed call. That's provided for you: just use it for anything that runs on the GPU.

In [ ]:
def measure_gpu(fn, n_warmup=20, n_iter=200):
    """Like `measure`, but waits for the GPU to finish each call. Returns per-call times in µs."""
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize()
    t = np.empty(n_iter)
    for i in range(n_iter):
        t0 = time.perf_counter_ns(); fn(); torch.cuda.synchronize(); t[i] = time.perf_counter_ns() - t0
    return t / 1e3

if not HAS_GPU:
    print("No GPU in this runtime: skip Section 5c, or switch runtime and rerun.")
else:
    DEV = "cuda"
    def gpu_matmul_tflops(n=4096, dtype=torch.float32, reps=10):
        a, b = torch.randn(n, n, device=DEV, dtype=dtype), torch.randn(n, n, device=DEV, dtype=dtype)
        t = np.median(measure_gpu(lambda: a @ b, n_warmup=3, n_iter=reps))
        return 2 * n**3 / (t * 1e-6) / 1e12
    def gpu_triad_gbs(n=64 * 2**20, reps=10):
        b, c = torch.randn(n, device=DEV), torch.randn(n, device=DEV); a = torch.empty(n, device=DEV)
        t = np.median(measure_gpu(lambda: torch.add(b, c, alpha=1.5, out=a), n_warmup=3, n_iter=reps))
        return 3 * 4 * n / (t * 1e-6) / 1e9
    P_gpu, P_gpu16, beta_gpu = gpu_matmul_tflops() * 1e12, gpu_matmul_tflops(dtype=torch.float16) * 1e12, gpu_triad_gbs() * 1e9
    print(f"GPU: matmul fp32 {P_gpu/1e12:.1f} TFLOP/s | fp16 {P_gpu16/1e12:.1f} TFLOP/s | bandwidth {beta_gpu/1e9:.0f} GB/s | "
          f"ridge (fp32) {P_gpu/beta_gpu:.0f} FLOP/byte")
    print(f"CPU: matmul {P_peak/1e9:.0f} GFLOP/s | bandwidth {beta/1e9:.0f} GB/s | ridge {P_peak/beta:.1f} FLOP/byte")

In [ ]:
if HAS_GPU:
    gmodel = JetMLP().to(DEV); gmodel.load_state_dict(model.state_dict()); gmodel.eval()   # a copy of your trained model, on the GPU
    gbatches = [1, 16, 256, 4096, 65536, 262144]
    Xg = torch.from_numpy(np.resize(X_test, (max(gbatches), 16)).astype(np.float32)).to(DEV)   # data already on the GPU
    glat, gthr = [], []
    with torch.inference_mode():
        for B in gbatches:
            xb = Xg[:B]
            t = measure_gpu(lambda: gmodel(xb), n_warmup=10, n_iter=max(10, min(300, int(5e6 / B))))
            glat.append(np.median(t)); gthr.append(B / (np.median(t) * 1e-6))
            print(f"GPU B={B:>7}: {glat[-1]:9.1f} µs/batch  {gthr[-1]/1e6:9.2f} M samples/s")
    # the cost of getting one batch of inputs onto the GPU in the first place
    for B in [1, 65536]:
        xcpu = torch.from_numpy(np.resize(X_test, (B, 16)).astype(np.float32))
        print(f"copy B={B:>6} inputs CPU→GPU: {np.median(measure_gpu(lambda: xcpu.to(DEV))):8.1f} µs")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ax[0].semilogx(batches, np.array(thr) / 1e6, "o-", label="CPU"); ax[0].semilogx(gbatches, np.array(gthr) / 1e6, "s-", label="GPU")
    ax[0].set(xlabel="batch size", ylabel="throughput [M samples/s]", yscale="log", title="throughput"); ax[0].legend()
    I_ax = np.logspace(-2, 3, 200)
    ax[1].loglog(I_ax, np.minimum(P_peak, I_ax * beta) / 1e9, "k-", label="CPU roof")
    ax[1].loglog(I_ax, np.minimum(P_gpu, I_ax * beta_gpu) / 1e9, "k--", label="GPU roof (fp32)")
    ax[1].scatter([intensity(model, B) for B in batches], np.array(thr) * 2 * count_macs(model) / 1e9, label="jet MLP on CPU", zorder=3)
    ax[1].scatter([intensity(model, B) for B in gbatches], np.array(gthr) * 2 * count_macs(model) / 1e9, marker="s", label="jet MLP on GPU", zorder=3)
    ax[1].set(xlabel="arithmetic intensity [FLOP/byte]", ylabel="GFLOP/s", title="CPU and GPU rooflines"); ax[1].legend(fontsize=8)
    for a in ax: a.grid(alpha=.3, which="both")
    plt.tight_layout(); plt.show()

✋ **Pause 2.**

**Q4** From your Section 5c output and plots: (a) at batch 1, which is faster, the CPU or the GPU? (b) At roughly what batch size does the GPU overtake the CPU in throughput?


📌 **What this shows** (read after answering Q4).
- **The GPU's ridge point is far to the right:** it needs many operations per byte to stay busy. This tiny model's intensity is capped at ~100 (Section 2), so it can never fill a GPU, whatever the batch size.
- **At batch 1, the GPU is overhead-bound too.** Every layer is a separate launch costing µs, plus the wait for the result.
- **The GPU wins at large batches.** That's the telescope or offline-processing regime, not the trigger's.
- **fp16 beats fp32 on the GPU**, thanks to its tensor cores. Lab 2b comes back to this.
- **Just copying the inputs onto the GPU can cost as much as computing on them.** An accelerator only helps if the data is already there, which is why triggers put FPGAs directly on the detector links.

## 6. 🟥 AI-off: back to the trigger budget (10 min)  ✋ Pause 3
The LHC Level-1 trigger (ATLAS or CMS) sees a bunch crossing every **25 ns** (40 MHz). A neural network running there might get on the order of
**100 ns** of latency and must accept a new input every 25 ns.

**Q5** What sustained FLOP/s does running this MLP on every crossing need? Is it above or below your measured $P_\text{peak}$?

**Q6** So compute isn't the problem. Give **three** distinct reasons why neither your CPU nor your GPU could do this job. Use your measurements from Sections 3 and 5c.



📌 **What an FPGA does instead.** At 200 MHz, a 100 ns budget is just **20 clock cycles**, about 5 per layer. That's only possible with every multiplication built as its own circuit,
working in parallel, with a new input accepted every cycle (5 ns, comfortably inside 25 ns). Day 7 and Lab 4 show how.

## 7. Wrap-up
Save your key plots (the batch sweep and the rooflines), and record your numbers.
**Assignment A1** builds on these exact numbers.

| My Colab machine | CPU | GPU |
|---|---|---|
| model | | |
| $P_\text{peak}$ (fp32) | GFLOP/s | TFLOP/s |
| $\beta$ [GB/s] | | |
| ridge point [FLOP/B] | | |
| jet MLP batch-1 median / p99 latency [µs] | | |
| best throughput [M samples/s] at B = | | |